> 📒 이 노트북은 [펭귄 데이터로 배우는 인공지능](https://yanghyeokjin33.github.io/Algorithm/) 수업의 실습이에요. 원본: [https://github.com/YANGHYEOKJIN33/Algorithm](https://github.com/YANGHYEOKJIN33/Algorithm)

# 🔍 결측치와 이상치 찾기

**목표**: 결측치의 여부·개수·위치를 확인하고, 사분위수(IQR)와 상자그림으로 이상치를 찾는다.

연습 데이터에는 수업을 위해 숨겨 둔 함정이 있어요: **빈칸**, **이상한 값**, **겹친 행**. 하나씩 찾아봐요!

> **Colab 사용법** — 셀 왼쪽의 ▶ 단추(또는 **Shift + Enter**)를 누르면 그 셀이 실행돼요.
> 위에서부터 **차례로** 실행하세요. 고쳐 보고 싶으면 먼저 **파일 → 드라이브에 사본 저장**을 눌러 내 것으로 만드세요.
> 파이썬을 몰라도 괜찮아요. 코드 위의 설명과 사이트의 의사코드를 짝지어 읽으면 돼요.

- 수업용 펭귄 데이터를 인터넷 주소에서 바로 불러와요. 앞 시간에 크롤링한 파일과 같은 내용이에요.

In [ ]:
import pandas as pd
URL = 'https://raw.githubusercontent.com/YANGHYEOKJIN33/Algorithm/main/data/penguins.csv'
df = pd.read_csv(URL)          # 크롤링한 penguins.csv를 올렸다면 pd.read_csv('penguins.csv')
print(df.shape)
df.head()

## 1. 결측치 여부 — `isnull()`

칸마다 비었으면 **True**, 값이 있으면 **False**인 표를 만들어요.

- 결과표는 원래 표와 크기가 같아요. True인 칸이 결측치예요(인덱스 3, 6, 8~11 행을 보세요).

In [ ]:
df.isnull().head(12)

## 2. 결측치 개수 — `isnull().sum()`

True는 1, False는 0으로 계산되니 **더하면 개수**가 돼요.

- 열마다 빈칸이 몇 개인지 보여 줘요. 성별이 가장 많이 비었어요.

In [ ]:
print(df.isnull().sum())
print('전체 결측치:', df.isnull().sum().sum())

## 3. 결측치 위치 — 인덱스 찾기

행마다(`axis=1`) True가 하나라도 있는지(`any`) 보고, 그런 행의 인덱스를 모아요. 인덱스는 **0부터** 세는 자리 번호예요.

- df[조건]은 조건이 True인 행만 골라요.
- df.loc[인덱스 목록]은 그 행들을 보여 줘요.

In [ ]:
has_nan = df.isnull().any(axis=1)
where = df[has_nan].index
print('결측치가 있는 행의 인덱스:', list(where))
df.loc[where]

- df['몸무게'].isnull()은 몸무게 열만 검사해요.

In [ ]:
# 열 하나만 보고 싶을 때
df[df['몸무게'].isnull()]

## 4. 이상치 — 사분위수와 울타리

    Q1 = 25% 자리 값, Q3 = 75% 자리 값, IQR = Q3 − Q1
    아래 울타리 = Q1 − 1.5 × IQR,  위 울타리 = Q3 + 1.5 × IQR
    울타리 밖의 값 → 이상치

- quantile(0.25)는 25% 자리(Q1)의 값이에요.
- | 는 "또는"이라는 뜻이에요. 아래 울타리보다 작거나, 위 울타리보다 큰 행을 골라요.
- 몸무게 8200g인 아델리펭귄이 나와요 — 3200을 잘못 적은 값이에요!

In [ ]:
s = df['몸무게']
q1 = s.quantile(0.25)
q3 = s.quantile(0.75)
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr
print(f'Q1={q1}, Q3={q3}, IQR={iqr}')
print(f'울타리: {low} ~ {high}')
df[(s < low) | (s > high)]          # 이상치인 행

## 5. 상자그림으로 한눈에

상자(Q1~Q3), 가운데 선(중앙값), 수염, 그리고 **수염 밖의 점 = 이상치**.

- 그래프에 한글(제목·축 이름)이 깨지지 않도록 글꼴 도구를 설치해요. 처음 한 번 1~2분 걸릴 수 있어요.

In [ ]:
!pip install -q koreanize-matplotlib
try:
    import koreanize_matplotlib  # 그래프에 한글 글꼴을 입혀 줘요
except ImportError:
    print('한글 글꼴을 불러오지 못했어요. 그래프 글자가 네모로 보일 수 있어요.')
import matplotlib.pyplot as plt

- dropna()로 빈칸을 빼고 그려요(빈칸은 그릴 수 없어요).
- 맨 위에 홀로 떨어진 점이 8200g이에요. 사이트의 "실제 비율로" 그림과 같은 모양이에요.

In [ ]:
plt.figure(figsize=(4, 5))
plt.boxplot(df['몸무게'].dropna())
plt.title('펭귄 몸무게 상자그림')
plt.ylabel('몸무게 (g)')
plt.show()

- by='종'은 종마다 상자를 따로 그려요. 젠투가 가장 무겁다는 것도 보여요.

In [ ]:
# 여러 열을 한 번에 — 종별로 나눠 보면 더 잘 보여요
df.boxplot(column='몸무게', by='종', figsize=(6, 4))
plt.suptitle('')
plt.title('종별 몸무게')
plt.show()

## 6. 사이트와 같은 13마리로 확인

사이트 "이상치 ① 사분위수"의 아델리펭귄 13마리로 계산하면 Q1=3450, Q2=3700, Q3=3800이 나와요.

- isin(목록)은 번호가 목록 안에 있는 행만 골라요.

In [ ]:
ids = [1, 2, 3, 5, 6, 7, 10, 11, 12, 13, 14, 16, 17]
small = df[df['번호'].isin(ids)].drop_duplicates()['몸무게']
print(small.quantile([0.25, 0.5, 0.75]))

## 🧩 확인해 봐요 · 한 번에 보기

- `df.describe()`를 실행하면 개수(count)·평균·사분위수·최솟값·최댓값이 한 번에 나와요. count가 345보다 작은 열은 왜 그럴까요?
- `df.duplicated().sum()`을 실행해 보세요. 겹친 행이 몇 개인가요? → 다음 시간 🧹 전처리에서 지워요.

- describe()는 숫자 열의 요약표예요.

In [ ]:
print('겹친 행:', df.duplicated().sum())
df.describe()